<a href="https://colab.research.google.com/github/snehacs24-png/BIS-lab/blob/main/Untitled6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import cv2
import numpy as np
import matplotlib.pyplot as plt
import random

random.seed(42)
np.random.seed(42)

image_path = "/content/your_image.jpg"

image = cv2.imread(image_path)

if image is None:
    raise ValueError("Image not found. Check the file path.")

image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
image = cv2.resize(image, (512, 512))

gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)

population_size = 30
generations = 50
num_thresholds = 3
mutation_rate = 0.1
crossover_rate = 0.8

def fitness(chromosome):
    thresholds = np.sort(chromosome).astype(int)
    pixels = gray.flatten()
    total_mean = np.mean(pixels)
    variance = 0

    boundaries = [0] + list(thresholds) + [256]

    for i in range(len(boundaries) - 1):
        group = pixels[
            (pixels >= boundaries[i]) &
            (pixels < boundaries[i + 1])
        ]

        if len(group) > 0:
            probability = len(group) / len(pixels)
            mean = np.mean(group)
            variance += probability * (mean - total_mean) ** 2

    return variance

population = np.random.randint(
    1, 255, (population_size, num_thresholds)
)

best_fitness_history = []

for generation in range(generations):

    scores = np.array([fitness(ind) for ind in population])

    sorted_indices = np.argsort(scores)[::-1]

    population = population[sorted_indices]

    best_fitness_history.append(scores[sorted_indices[0]])

    new_population = [population[0].copy()]

    while len(new_population) < population_size:

        p1 = population[random.randint(0, 9)]
        p2 = population[random.randint(0, 9)]

        if random.random() < crossover_rate:
            point = random.randint(1, num_thresholds - 1)
            child = np.concatenate((p1[:point], p2[point:]))
        else:
            child = p1.copy()

        for i in range(num_thresholds):
            if random.random() < mutation_rate:
                child[i] = random.randint(1, 255)

        child = np.sort(child)
        new_population.append(child)

    population = np.array(new_population)

best_thresholds = np.sort(population[0]).astype(int)

segmented = np.digitize(gray, best_thresholds)

plt.figure(figsize=(12, 8))

plt.subplot(2, 2, 1)
plt.imshow(image)
plt.title("Original Image")
plt.axis("off")

plt.subplot(2, 2, 2)
plt.imshow(gray, cmap="gray")
plt.title("Grayscale Image")
plt.axis("off")

plt.subplot(2, 2, 3)
plt.imshow(segmented, cmap="gray")
plt.title("GA Segmented Image")
plt.axis("off")

plt.subplot(2, 2, 4)
plt.plot(best_fitness_history)
plt.title("GA Convergence Graph")
plt.xlabel("Generation")
plt.ylabel("Fitness Value")
plt.grid()

plt.tight_layout()
plt.show()

print("Optimal Thresholds:", best_thresholds)
print("Best Fitness:", max(best_fitness_history))
print("Segmentation Completed Successfully")
